<a href="https://colab.research.google.com/github/kamilal-hub/DSA210-TERM-PROJECT/blob/main/dsa210.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://datasets.imdbws.com/title.basics.tsv.gz
!wget https://datasets.imdbws.com/title.ratings.tsv.gz

In [ ]:
import pandas as pd

basics = pd.read_csv('title.basics.tsv.gz', sep='\t', low_memory=False, na_values='\\N')
ratings = pd.read_csv('title.ratings.tsv.gz', sep='\t', low_memory=False, na_values='\\N')

tv_series = basics[basics['titleType'] == 'tvSeries']

merged = pd.merge(tv_series, ratings, on='tconst')

merged['startYear'] = pd.to_numeric(merged['startYear'], errors='coerce')
final_list = merged[(merged['startYear'] >= 2010) & (merged['startYear'] <= 2025)]

top_250 = final_list.sort_values(by='numVotes', ascending=False).head(250)


print(f"Successfully filtered {len(top_250)} series.")
top_250[['primaryTitle', 'averageRating', 'numVotes', 'startYear']].head(10)

In [ ]:
import pandas as pd
from pytrends.request import TrendReq
import time
import random

pytrends = TrendReq(hl='en-US', tz=360)


series_list = top_250['primaryTitle'].head(50).tolist()
all_trends = []

print("Starting data collection from Google Trends...")

for name in series_list:
    try:
        pytrends.build_payload([name], timeframe='2010-01-01 2025-12-31')
        df = pytrends.interest_over_time()
        if not df.empty:

            series_trend = df[name].reset_index()
            series_trend['title'] = name
            all_trends.append(series_trend)
            print(f"--- {name} successfully fetched.")
        time.sleep(random.uniform(7, 12))
    except Exception as e:
        print(f"!!! Error fetching {name}: {e}")
        time.sleep(60)
        continue
if all_trends:
    final_trends_df = pd.concat(all_trends, ignore_index=True)
    print("Data collection complete!")

In [ ]:
final_trends_df.to_csv('tv_series_google_trends.csv', index=False)
print("Data successfully saved to 'tv_series_google_trends.csv'!")

In [ ]:

peak_interests = []
for title in final_trends_df['title'].unique():
    peak_val = final_trends_df[final_trends_df['title'] == title][title].max()
    peak_interests.append({'primaryTitle': title, 'peak_search': peak_val})

summary_trends = pd.DataFrame(peak_interests)
combined_df = pd.merge(top_250, summary_trends, on='primaryTitle')

print(f"Merged data for {len(combined_df)} series.")
combined_df[['primaryTitle', 'averageRating', 'peak_search']].head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.regplot(data=combined_df, x='averageRating', y='numVotes', color='teal', scatter_kws={'alpha':0.5})
plt.title('Relationship Between IMDb Rating and Audience Size', fontsize=14)
plt.xlabel('IMDb Rating', fontsize=12)
plt.ylabel('Total Number of Votes (numVotes)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
correlation = combined_df['averageRating'].corr(combined_df['numVotes'])

print("-" * 45)
print("HYPOTHESIS TEST RESULTS ")
print(f"Correlation between Rating and Popularity: {correlation:.2f}")
print("-" * 45)

if correlation > 0.4:
    print("Interpretation: There is a significant positive correlation. Quality often leads to popularity.")
else:
    print("Interpretation: The correlation is weak. A high rating does not guarantee mass-market popularity.")